In [1]:
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, LinearSegmentedColormap
from matplotlib.ticker import LogFormatterMathtext
import matplotlib.ticker as ticker
from scipy.ndimage import gaussian_filter
import numpy as np
import pandas as pd
import dask
import zarr
import xarray as xr

In [2]:
llc_path = '/orcd/data/abodner/002/cody/LLC_patch/LLC4320_face1_i2880-3600_j720-1440.zarr'
llc_patch_full = xr.open_dataset(llc_path, consolidated=True)

# EXPERIMENT 1
#epochs=1, steps=1, vars=all, loss=mse_diff_weighted, norm=syncbatchnorm, padding='constant,' pred_residual=true,
emulator_1_path = '/orcd/data/abodner/002/cody/inference_patch/config_tests_4-23-26/2026-04-23-eval:Samudra_LLC:config_tests_experiment_1/predictions_4d.zarr'
emulator_1_patch_full = xr.open_dataset(emulator_1_path, consolidated=True) 

# EXPERIMENT 2
# epochs=1, steps=1, vars=all, loss=mse_diff_weighted, norm=group_norm_32, padding='constant,' pred_residual=true,
emulator_2_path = '/orcd/data/abodner/002/cody/inference_patch/config_tests_4-23-26/2026-04-23-eval:Samudra_LLC:config_tests_experiment_2/predictions_4d.zarr'
emulator_2_patch_full = xr.open_dataset(emulator_2_path, consolidated=True) 

# EXPERIMENT 3
# epochs=1, steps=1, vars=all, loss=mse_diff_weighted + dynamically weighted loss, norm=syncbatchnorm, padding='constant,' pred_residual=true,
# emulator_3_path = '/orcd/data/abodner/002/cody/inference_patch/config_tests_4-23-26/2026-04-23-eval:Samudra_LLC:config_tests_experiment_3/predictions_4d.zarr'
# emulator_3_patch_full = xr.open_dataset(emulator_3_path, consolidated=True) 

# EXPERIMENT 4
# epochs=1, steps=1, vars=all, loss=mse_diff_weighted + dynamically weighted loss, norm=syncbatchnorm, padding='halo_sponge,' pred_residual=true
#emulator_4_path = '/orcd/data/abodner/002/cody/inference_patch/config_tests_4-23-26/2026-04-23-eval:Samudra_LLC:config_tests_experiment_4/predictions_4d.zarr'
#emulator_4_patch_full = xr.open_dataset(emulator_4_path, consolidated=True) 

# EXPERIMENT 5
# epochs=1, steps=1, vars=all, loss=mse_diff_weighted + weighted loss [U,V = 1.0, Theta, Salt, Eta = 1.5 , norm=group_norm_32, padding='constant,' pred_residual=true,
emulator_3_path = '/orcd/data/abodner/002/cody/inference_patch/config_tests_4-23-26/2026-04-24-eval:Samudra_LLC:config_tests_experiment_5/predictions_4d.zarr'
emulator_3_patch_full = xr.open_dataset(emulator_3_path, consolidated=True) 

# EXPERIMENT 6
# epochs=1, steps=1, vars=all, loss=mae gradient, norm=group_norm_32, padding='constant,' pred_residual=true,
emulator_4_path = '/orcd/data/abodner/002/cody/inference_patch/config_tests_4-23-26/2026-04-24-eval:Samudra_LLC:config_tests_experiment_6/predictions_4d.zarr'
emulator_4_patch_full = xr.open_dataset(emulator_4_path, consolidated=True) 

In [3]:
emulator_times = emulator_1_patch_full.time.values
matching_times = pd.DatetimeIndex([
    pd.Timestamp(t.year, t.month, t.day, t.hour, t.minute, t.second)
    for t in emulator_times
])

llc_patch = llc_patch_full.sel(time=matching_times)

In [4]:
emulator_1_patch = emulator_1_patch_full
emulator_2_patch = emulator_2_patch_full
emulator_3_patch = emulator_3_patch_full
emulator_4_patch = emulator_4_patch_full

In [5]:
def format_time(t_val):
    """Format a time value to DD/MM/YYYY:HH regardless of cftime or datetime64."""
    try:
        # cftime objects
        return f"{t_val.day:02d}/{t_val.month:02d}/{t_val.year}:{t_val.hour:02d}h"
    except AttributeError:
        # numpy datetime64
        t_pd = pd.Timestamp(t_val)
        return f"{t_pd.day:02d}/{t_pd.month:02d}/{t_pd.year}:{t_pd.hour:02d}h"


In [6]:
emulator_1_patch['XC'] = llc_patch['XC']
emulator_1_patch['YC'] = llc_patch['YC']
emulator_1_patch['rA'] = llc_patch['rA']
emulator_1_patch['Z'] = llc_patch['Z']

emulator_2_patch['XC'] = llc_patch['XC']
emulator_2_patch['YC'] = llc_patch['YC']
emulator_2_patch['rA'] = llc_patch['rA']
emulator_2_patch['Z'] = llc_patch['Z']

emulator_3_patch['XC'] = llc_patch['XC']
emulator_3_patch['YC'] = llc_patch['YC']
emulator_3_patch['rA'] = llc_patch['rA']
emulator_3_patch['Z'] = llc_patch['Z']

emulator_4_patch['XC'] = llc_patch['XC']
emulator_4_patch['YC'] = llc_patch['YC']
emulator_4_patch['rA'] = llc_patch['rA']
emulator_4_patch['Z'] = llc_patch['Z']

In [7]:
omega = 7.2921e-5

all_patches = {
    'llc': llc_patch,
    'emulator_1': emulator_1_patch,
    'emulator_2': emulator_2_patch,
    'emulator_3': emulator_3_patch,
    'emulator_4': emulator_4_patch
}

for patch_name, patch in all_patches.items():
    print(f"Computing vorticity for {patch_name}...")
    
    f_0 = np.abs(2 * omega * np.sin(np.deg2rad(patch['YC'].values)))
    
    dx = np.sqrt(patch['rA'].values)  # (j, i) in meters
    dy = dx.copy()
    
    U = patch['U'].values
    V = patch['V'].values
    
    dvdx = (np.roll(V, -1, axis=3) - np.roll(V, 1, axis=3)) / (2 * dx[np.newaxis, np.newaxis, :, :])
    dudy = (np.roll(U, -1, axis=2) - np.roll(U, 1, axis=2)) / (2 * dy[np.newaxis, np.newaxis, :, :])
    
    vort = dvdx - dudy
    vort_normalized = vort / f_0[np.newaxis, np.newaxis, :, :]
    
    patch['vorticity'] = (('time', 'k', 'j', 'i'), vort_normalized)
    print(f"  ✓ vorticity: {vort_normalized.shape}")

print("Done computing vorticity!")

Computing vorticity for llc...
  ✓ vorticity: (10, 51, 720, 720)
Computing vorticity for emulator_1...
  ✓ vorticity: (10, 51, 720, 720)
Computing vorticity for emulator_2...
  ✓ vorticity: (10, 51, 720, 720)
Computing vorticity for emulator_3...
  ✓ vorticity: (10, 51, 720, 720)
Computing vorticity for emulator_4...
  ✓ vorticity: (10, 51, 720, 720)
Done computing vorticity!


In [8]:
omega = 7.2921e-5

for patch_name, patch in all_patches.items():
    print(f"Computing strain for {patch_name}...")
    
    f_0 = np.abs(2 * omega * np.sin(np.deg2rad(patch['YC'].values)))
    
    dx = np.sqrt(patch['rA'].values)  # (j, i) in meters
    dy = dx.copy()
    
    U = patch['U'].values
    V = patch['V'].values
    
    u_x = (np.roll(U, -1, axis=3) - np.roll(U, 1, axis=3)) / (2 * dx[np.newaxis, np.newaxis, :, :])
    u_y = (np.roll(U, -1, axis=2) - np.roll(U, 1, axis=2)) / (2 * dy[np.newaxis, np.newaxis, :, :])
    v_x = (np.roll(V, -1, axis=3) - np.roll(V, 1, axis=3)) / (2 * dx[np.newaxis, np.newaxis, :, :])
    v_y = (np.roll(V, -1, axis=2) - np.roll(V, 1, axis=2)) / (2 * dy[np.newaxis, np.newaxis, :, :])
    
    sigma_n = u_x - v_y
    sigma_s = v_x + u_y
    sigma = np.sqrt(sigma_n**2 + sigma_s**2)
    
    sigma_normalized = sigma / f_0[np.newaxis, np.newaxis, :, :]
    
    patch['strain'] = (('time', 'k', 'j', 'i'), sigma_normalized)
    print(f"  ✓ strain: {sigma_normalized.shape}")

print("Done computing strain!")



Computing strain for llc...
  ✓ strain: (10, 51, 720, 720)
Computing strain for emulator_1...
  ✓ strain: (10, 51, 720, 720)
Computing strain for emulator_2...
  ✓ strain: (10, 51, 720, 720)
Computing strain for emulator_3...
  ✓ strain: (10, 51, 720, 720)
Computing strain for emulator_4...
  ✓ strain: (10, 51, 720, 720)
Done computing strain!


In [ ]:
# ============== VORTICITY AND STRAIN SURFACE PLOTS ==============
vars = ['vorticity', 'strain']
colormaps = {'vorticity': 'magma', 'strain': 'plasma'}
# ================================================================

for var in vars:
    print(f"Generating plots for {var}...")
    
    
    n_times = len(emulator_1_patch.time)
    time_step = 1
    time_indices = list(range(0, n_times, time_step))
    nrows = len(time_indices)
    ncols = 5
    
    cmap = colormaps[var]
    
    # ==================== PLOT 1: Surface fields ====================
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 3*nrows), dpi=200)
    
    if nrows == 1:
        axes = axes.reshape(1, -1)
    
    for row, t in enumerate(time_indices):
        time_str = format_time(emulator_1_patch.time.values[t])
        
        llc_vis = llc_patch.isel(time=t, k=0)[var]
        emulator_1_vis = emulator_1_patch.isel(time=t, k=0)[var]
        emulator_2_vis = emulator_2_patch.isel(time=t, k=0)[var]
        emulator_3_vis = emulator_3_patch.isel(time=t, k=0)[var]
        emulator_4_vis = emulator_4_patch.isel(time=t, k=0)[var]
        
        vmin = np.min([llc_vis.values.min(), emulator_1_vis.values.min(), 
                       emulator_2_vis.values.min(), emulator_3_vis.values.min(),
                       emulator_4_vis.values.min()])
        vmax = np.max([llc_vis.values.max(), emulator_1_vis.values.max(), 
                       emulator_2_vis.values.max(), emulator_3_vis.values.max(),
                       emulator_4_vis.values.max()])
        
        ax1, ax2, ax3, ax4, ax5 = axes[row, 0], axes[row, 1], axes[row, 2], axes[row, 3], axes[row, 4]
        
        cf1 = ax1.contourf(llc_vis.coords.get('i', np.arange(llc_vis.shape[-1])), 
                           llc_vis.coords.get('j', np.arange(llc_vis.shape[-2])), llc_vis, 
                           cmap=cmap, vmin=vmin, vmax=vmax, levels=30)
        ax1.set_title(f'LLC {var} {time_str}', fontsize=8)
        plt.colorbar(cf1, ax=ax1)
        
        cf2 = ax2.contourf(emulator_1_vis.coords.get('i', np.arange(emulator_1_vis.shape[-1])), 
                           emulator_1_vis.coords.get('j', np.arange(emulator_1_vis.shape[-2])), emulator_1_vis, 
                           cmap=cmap, vmin=vmin, vmax=vmax, levels=30)
        ax2.set_title(f'Emulator 1 {var} {time_str}', fontsize=8)
        plt.colorbar(cf2, ax=ax2)
        
        cf3 = ax3.contourf(emulator_2_vis.coords.get('i', np.arange(emulator_2_vis.shape[-1])), 
                           emulator_2_vis.coords.get('j', np.arange(emulator_2_vis.shape[-2])), emulator_2_vis, 
                           cmap=cmap, vmin=vmin, vmax=vmax, levels=30)
        ax3.set_title(f'Emulator 2 {var} {time_str}', fontsize=8)
        plt.colorbar(cf3, ax=ax3)
        
        cf4 = ax4.contourf(emulator_3_vis.coords.get('i', np.arange(emulator_3_vis.shape[-1])), 
                           emulator_3_vis.coords.get('j', np.arange(emulator_3_vis.shape[-2])), emulator_3_vis, 
                           cmap=cmap, vmin=vmin, vmax=vmax, levels=30)
        ax4.set_title(f'Emulator 3 {var} {time_str}', fontsize=8)
        plt.colorbar(cf4, ax=ax4)
        
        cf5 = ax5.contourf(emulator_4_vis.coords.get('i', np.arange(emulator_4_vis.shape[-1])), 
                           emulator_4_vis.coords.get('j', np.arange(emulator_4_vis.shape[-2])), emulator_4_vis, 
                           cmap=cmap, vmin=vmin, vmax=vmax, levels=30)
        ax5.set_title(f'Emulator 4 {var} {time_str}', fontsize=8)
        plt.colorbar(cf5, ax=ax5)
    
    plt.tight_layout()
    plt.savefig(f'figs/mixing/{var}/surface_{var}_fields.png')
    plt.close()
    
    # ==================== PLOT 2: Difference fields ====================
    fig, axes = plt.subplots(nrows, ncols-1, figsize=(16, 3*nrows), dpi=200)
    
    if nrows == 1:
        axes = axes.reshape(1, -1)
    
    for row, t in enumerate(time_indices):
        time_str = format_time(emulator_1_patch.time.values[t])
        
        llc_vis = llc_patch.isel(time=t, k=0)[var]
        emulator_1_vis = emulator_1_patch.isel(time=t, k=0)[var]
        emulator_2_vis = emulator_2_patch.isel(time=t, k=0)[var]
        emulator_3_vis = emulator_3_patch.isel(time=t, k=0)[var]
        emulator_4_vis = emulator_4_patch.isel(time=t, k=0)[var]
        
        diff_1 = llc_vis.values - emulator_1_vis.values
        diff_2 = llc_vis.values - emulator_2_vis.values
        diff_3 = llc_vis.values - emulator_3_vis.values
        diff_4 = llc_vis.values - emulator_4_vis.values
        
        abs_max = np.max([np.abs(diff_1).max(), np.abs(diff_2).max(), 
                          np.abs(diff_3).max(), np.abs(diff_4).max()])
        vmin, vmax = -abs_max, abs_max
        
        ax1, ax2, ax3, ax4 = axes[row, 0], axes[row, 1], axes[row, 2], axes[row, 3]
        
        cf1 = ax1.contourf(llc_vis.coords.get('i', np.arange(llc_vis.shape[-1])), 
                           llc_vis.coords.get('j', np.arange(llc_vis.shape[-2])), diff_1, 
                           cmap="bwr", vmin=vmin, vmax=vmax, levels=30)
        ax1.set_title(f'LLC - Em1 {var} {time_str}', fontsize=8)
        
        cf2 = ax2.contourf(llc_vis.coords.get('i', np.arange(llc_vis.shape[-1])), 
                           llc_vis.coords.get('j', np.arange(llc_vis.shape[-2])), diff_2, 
                           cmap="bwr", vmin=vmin, vmax=vmax, levels=30)
        ax2.set_title(f'LLC - Em2 {var} {time_str}', fontsize=8)
        
        cf3 = ax3.contourf(llc_vis.coords.get('i', np.arange(llc_vis.shape[-1])), 
                           llc_vis.coords.get('j', np.arange(llc_vis.shape[-2])), diff_3, 
                           cmap="bwr", vmin=vmin, vmax=vmax, levels=30)
        ax3.set_title(f'LLC - Em3 {var} {time_str}', fontsize=8)
        
        cf4 = ax4.contourf(llc_vis.coords.get('i', np.arange(llc_vis.shape[-1])), 
                           llc_vis.coords.get('j', np.arange(llc_vis.shape[-2])), diff_4, 
                           cmap="bwr", vmin=vmin, vmax=vmax, levels=30)
        ax4.set_title(f'LLC - Em4 {var} {time_str}', fontsize=8)
        
        fig.colorbar(cf4, ax=[ax1, ax2, ax3, ax4], orientation='vertical', 
                     fraction=0.046, pad=0.04)
    
    plt.savefig(f'figs/mixing/{var}/surface_{var}_differences.png')
    plt.close()
    
    print(f"Saved plots for {var}")

In [ ]:
vars = ['vorticity', 'strain']
ref_lines = {
    'vorticity': [0.5, 1.0],  
    'strain': [0.5, 1.0]       
}

for var in vars:
    print(f"Generating depth error plots for {var}...")
    
    
    n_depths = llc_patch.sizes['k']
    n_times = len(emulator_1_patch.time)
    time_indices = list(range(n_times))
    
    nrows = len(time_indices)
    ncols = 4
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3*nrows), dpi=150)
    
    if nrows == 1:
        axes = axes.reshape(1, -1)
    
    emulator_patches_list = [
        ('Emulator 1', emulator_1_patch),
        ('Emulator 2', emulator_2_patch),
        ('Emulator 3', emulator_3_patch),
        ('Emulator 4', emulator_4_patch)
    ]
    
    depths = np.arange(n_depths)
    
    for row, t in enumerate(time_indices):
        time_str = format_time(emulator_1_patch.time.values[t])
        
        llc_data = llc_patch.isel(time=t)[var].values  # (k, j, i)
        
        # First pass: compute all errors for shared xlim
        row_mean_errors = []
        row_median_errors = []
        
        for emulator_name, emulator_patch in emulator_patches_list:
            emu_data = emulator_patch.isel(time=t)[var].values  # (k, j, i)
            diff = np.abs(llc_data - emu_data)  # (k, j, i)
            
            # All-pixel mean and median
            diff_flat = diff.reshape(n_depths, -1)
            mean_errors = np.nanmean(diff_flat, axis=1)
            median_errors = np.nanmedian(diff_flat, axis=1)
            
            row_mean_errors.append(mean_errors)
            row_median_errors.append(median_errors)
        
        # Shared x-axis limits for this row
        all_errors = np.concatenate(row_mean_errors + row_median_errors)
        xmin = 0
        xmax = np.nanmax(all_errors) * 1.05
        
        # Second pass: plot
        for col, (emulator_name, _) in enumerate(emulator_patches_list):
            ax = axes[row, col]
            
            mean_errors = row_mean_errors[col]
            median_errors = row_median_errors[col]
            
            # Mean (blue)
            ax.scatter(mean_errors, depths, color='blue', s=30, alpha=0.7, zorder=3)
            ax.plot(mean_errors, depths, color='blue', alpha=0.4, linewidth=1.5, label='Mean')
            
            # Median (red)
            ax.scatter(median_errors, depths, color='red', s=30, alpha=0.7, zorder=3)
            ax.plot(median_errors, depths, color='red', alpha=0.4, linewidth=1.5, label='Median')
            
            # Variable-specific reference lines
            for ref_val in ref_lines[var]:
                ax.axvline(x=ref_val, color='black', linestyle='--', linewidth=1.5, alpha=0.5, zorder=2)
            
            ax.set_title(f'{emulator_name} {var} {time_str}', fontsize=8)
            ax.set_xlabel('Abs Error', fontsize=7)
            ax.set_ylabel('Depth (k)', fontsize=7)
            ax.set_ylim(n_depths - 1, 0)
            ax.set_xlim(xmin, xmax)
            ax.grid(alpha=0.2)
            ax.tick_params(labelsize=6)
            
            if col == 0:
                ax.legend(fontsize=6, loc='lower right')
    
    plt.tight_layout()
    plt.savefig(f'figs/mixing/{var}/depth_error_by_time.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Saved depth error plots for {var}")

print("Done!")

In [9]:
for patch_name, patch in all_patches.items():
    print(f"Computing grad_T for {patch_name}...")
    
    Theta = patch['Theta'].values
    
    dx = np.sqrt(patch['rA'].values)  # (j, i) in meters
    dy = dx.copy()
    
    dT_di = (np.roll(Theta, -1, axis=3) - np.roll(Theta, 1, axis=3)) / (2 * dx[np.newaxis, np.newaxis, :, :])
    dT_dj = (np.roll(Theta, -1, axis=2) - np.roll(Theta, 1, axis=2)) / (2 * dy[np.newaxis, np.newaxis, :, :])
    
    grad_T = np.sqrt(dT_di**2 + dT_dj**2)
    
    patch['grad_T'] = (('time', 'k', 'j', 'i'), grad_T)
    print(f"  ✓ grad_T: {grad_T.shape}")

print("Done computing grad_T!")

Computing grad_T for llc...
  ✓ grad_T: (10, 51, 720, 720)
Computing grad_T for emulator_1...
  ✓ grad_T: (10, 51, 720, 720)
Computing grad_T for emulator_2...
  ✓ grad_T: (10, 51, 720, 720)
Computing grad_T for emulator_3...
  ✓ grad_T: (10, 51, 720, 720)
Computing grad_T for emulator_4...
  ✓ grad_T: (10, 51, 720, 720)
Done computing grad_T!


In [12]:
from scipy.ndimage import gaussian_filter
from matplotlib.colors import LogNorm
from matplotlib.ticker import LogFormatterMathtext
import os

print("Generating time-separated JPDFs...")

#os.makedirs('figs/JPDFs', exist_ok=True)

n_times = len(emulator_1_patch.time)
time_indices = list(range(n_times))
nrows = len(time_indices)
ncols = 5  # LLC + 4 emulators

fig, axes = plt.subplots(nrows, ncols, figsize=(20, 4*nrows), dpi=150)

if nrows == 1:
    axes = axes.reshape(1, -1)

patch_info = [
    ('LLC', llc_patch),
    ('Emulator 1', emulator_1_patch),
    ('Emulator 2', emulator_2_patch),
    ('Emulator 3', emulator_3_patch),
    ('Emulator 4', emulator_4_patch)
]

for row, t in enumerate(time_indices):
    time_str = format_time(emulator_1_patch.time.values[t])
    print(f"  Processing time {t}: {time_str}")
    
    # First pass: compute all JPDFs for this time to get shared colorbar limits
    row_gradT_data = []
    
    for patch_name, patch in patch_info:
        # Extract data for upper 100m (k=0:20) at this time step
        vort = patch.vorticity.isel(time=t).values[:21]  # (k, j, i)
        strain = patch.strain.isel(time=t).values[:21]
        grad_T = patch.grad_T.isel(time=t).values[:21]
        
        v = vort.flatten()
        s = strain.flatten()
        g = grad_T.flatten()
        
        mask = ~(np.isnan(v) | np.isnan(s) | np.isnan(g)) & (g > 0)
        v = v[mask]; s = s[mask]; g = g[mask]
        
        # Subsample for performance
        idx = np.random.choice(len(v), min(100000, len(v)), replace=False)
        v_sub = v[idx]; s_sub = s[idx]; g_sub = g[idx]
        
        # Compute 2D histogram
        counts, xedges, yedges = np.histogram2d(v_sub, s_sub, bins=200)
        g_sum, _, _ = np.histogram2d(v_sub, s_sub, bins=[xedges, yedges], weights=g_sub)
        
        counts_smooth = gaussian_filter(counts, sigma=0.75)
        g_sum_smooth = gaussian_filter(g_sum, sigma=0.75)
        
        g_mean = np.full_like(counts_smooth, np.nan)
        valid = counts_smooth > 0
        g_mean[valid] = g_sum_smooth[valid] / counts_smooth[valid]
        g_mean = np.where(g_mean > 0, g_mean, np.nan)
        
        row_gradT_data.append((v_sub, s_sub, g_sub, g_mean, xedges, yedges))
    
    # Shared colorbar limits for this row
    shared_vmin_g = min([np.nanpercentile(d[3], 1) for d in row_gradT_data])
    shared_vmax_g = max([np.nanpercentile(d[3], 99) for d in row_gradT_data])
    log_min_g = np.floor(np.log10(shared_vmin_g))
    log_max_g = np.ceil(np.log10(shared_vmax_g))
    levels_g = 10 ** np.arange(log_min_g, log_max_g + 0.5, 0.5)
    
    # Second pass: plot
    for col, (patch_name, _) in enumerate(patch_info):
        ax = axes[row, col]
        v_sub, s_sub, g_sub, g_mean, xedges, yedges = row_gradT_data[col]
        
        xc = (xedges[:-1] + xedges[1:]) / 2
        yc = (yedges[:-1] + yedges[1:]) / 2
        
        cf = ax.contourf(xc, yc, g_mean.T, levels=levels_g,
                         cmap='magma_r', norm=LogNorm(vmin=shared_vmin_g, vmax=shared_vmax_g))
        ax.scatter(v_sub, s_sub, s=0.1, alpha=0.05, color='k', rasterized=True)
        
        lim = max(np.abs(v_sub).max(), s_sub.max())
        ax.plot([0, lim], [0, lim], 'k--', linewidth=1, label=r'$\sigma = |\zeta|$')
        ax.plot([0, -lim], [0, lim], 'k--', linewidth=1)
        
        ax.set_xlabel(r'$\zeta / f_0$', fontsize=8)
        ax.set_ylabel(r'$\sigma / |f_0|$', fontsize=8)
        ax.set_title(f'{patch_name} {time_str}', fontsize=9)
        ax.tick_params(labelsize=7)
        
        if col == 0:
            ax.legend(fontsize=6)
        
        # Add colorbar for last column of each row
        if col == ncols - 1:
            cbar = plt.colorbar(cf, ax=axes[row, :].tolist(), 
                               label=r'$|\nabla T|$', fraction=0.046, pad=0.04)
            cbar.ax.yaxis.set_major_formatter(LogFormatterMathtext())
            cbar.ax.tick_params(labelsize=7)

#plt.tight_layout()
plt.savefig('figs/JPDFs/JPDF_vort_strain_gradT_by_time.png', dpi=150, bbox_inches='tight')
plt.close()
#
print("✓ Saved time-separated JPDF figure")

Generating time-separated JPDFs...
  Processing time 0: 02/10/2012:00h
  Processing time 1: 02/10/2012:06h
  Processing time 2: 02/10/2012:12h
  Processing time 3: 02/10/2012:18h
  Processing time 4: 03/10/2012:00h
  Processing time 5: 03/10/2012:06h
  Processing time 6: 03/10/2012:12h
  Processing time 7: 03/10/2012:18h
  Processing time 8: 04/10/2012:00h
  Processing time 9: 04/10/2012:06h
✓ Saved time-separated JPDF figure
